In [0]:
# Project : AI Data Ingestion: Dataload Project
# Pipeline: `landing/` -> `data/processing/` -> profile + Gemini classification + schema -> `data/output/` -> `data/archive/`


# **Prerequisite:** run the project setup notebook first (volume, folders, `gemini` secret scope).
%pip install google-genai openpyxl

# MAGIC ## 1. Project configuration

import json
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "AI_DataIngestion"

PROJECT_FOLDER = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"
PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"

SECRET_SCOPE = "gemini"
SECRET_KEY = "GEMINI_API_KEY"
MODEL = "gemini-3.5-flash-lite"

# Create folders if they don't exist
for folder in [LANDING, PROCESSING, ARCHIVE, ERROR, OUTPUT]:
    dbutils.fs.mkdirs(str(folder))

# Enable MLflow tracing for every Gemini call (run BEFORE creating the client).
# Traces capture prompts, responses, latency, and token usage in the notebook's
# MLflow experiment. mlflow is preinstalled on this compute; no pip install needed.
import mlflow

mlflow.gemini.autolog()

# Initialize the AI client (the key comes from Databricks secrets, never printed)
client = genai.Client(api_key=dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY))

print("Folders ready and AI client initialized!")

## 2. Function: `load_dataframe`
def move_file(src, dst):
    """Move a file between volume folders."""
    dbutils.fs.mv(str(src), str(dst))


def load_dataframe(file_path):
    extension = file_path.suffix.lower()

    if extension == ".csv":
        return pd.read_csv(file_path)
    if extension == ".json":
        return pd.read_json(file_path)
    if extension == ".xlsx":
        return pd.read_excel(file_path)
    if extension == ".txt":
        return pd.read_csv(file_path, sep="|")

    print(f"Unsupported file type: {extension}")
    return None

# TEST IT: upload employee.csv into the landing folder first (Catalog > Volumes > landing > Upload)
display(dbutils.fs.ls(str(LANDING)))

test_file = LANDING / "employee.csv"  # change to your actual file name
df = load_dataframe(test_file) if test_file.exists() else None

if df is not None:
    print("\n--- Testing load_dataframe ---")
    display(df.head())
else:
    print(f"\nSkipping tests: {test_file.name} is not in landing/ (it may have already been processed).")


## 3. Function: `profile_dataframe`
def profile_dataframe(df, file_name):
    row_count = len(df)
    if row_count:
        null_percentage = (df.isnull().sum() / row_count * 100).round(2)
    else:
        null_percentage = df.isnull().sum()

    column_profile = pd.DataFrame({
        "column_name": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "null_count": df.isnull().sum().values,
        "actual_count": row_count,
        "unique_count": df.nunique().values,
        "null_percentage": null_percentage.values
    })

    profile_file = OUTPUT / f"{Path(file_name).stem}_profile.csv"
    column_profile.to_csv(profile_file, index=False)
    data_types = df.dtypes.astype(str).to_dict()
    duplicates = df.duplicated().sum()

    return data_types, column_profile, duplicates

if df is not None:
    print("--- Testing profile_dataframe ---")
    data_types, column_profile, duplicates = profile_dataframe(df, test_file.name)

    print("Data Types Dictionary:")
    print(data_types)
    print(f"\nDuplicate rows: {duplicates}")
    print("\nColumn Profile DataFrame:")
    display(column_profile)

## 4. Function: `build_prompts`
def build_prompts(file_name, df, data_types, column_profile):
    sample_json = df.head(10).astype(str).to_json(orient="records", indent=2, force_ascii=False)
    columns = df.columns.tolist()
    types_json = json.dumps(data_types, indent=2, ensure_ascii=False)
    profile_text = column_profile.to_string(index=False)

    classification_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Data governance domain, PII/confidential tags, automated data quality rules & checks. Return valid JSON."""

    schema_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Target analytical DDL schema, primary/foreign key evidence & data quality constraints. Return valid JSON."""
    return classification_prompt, schema_prompt

# TEST IT
if df is not None:
    print("--- Testing build_prompts ---")
    c_prompt, s_prompt = build_prompts(test_file.name, df, data_types, column_profile)

    print("Classification Prompt preview:")
    print(c_prompt[:500] + "...\n[TRUNCATED FOR DISPLAY]")

## 5. Function: `ask_gemini`
def ask_gemini(prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        # forces clean JSON (no ```json fences), so the saved .json files are valid
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    return response.text

# TEST IT
if df is not None:
    print("--- Testing ask_gemini ---")
    print("Sending prompt to AI... please wait.")
    classification_result = ask_gemini(c_prompt)
    print(classification_result)

## 6. Pipeline: `process_file` and `main`

def process_file(source_file):
    """
    End-to-end pipeline for one incoming file:
    1. Move landing/ -> data/processing/ (lock)
    2. Load into pandas (.csv, .json, .xlsx, .txt)
    3. Profile the data and save _profile.csv
    4. Build the governance and schema prompts
    5. Ask Gemini twice (classification + DDL schema)
    6. Save _classification.json, _schema.json and _processed.csv
    7. Archive the source file
    Any failure moves the file to data/error/.
    """
    file_name = source_file.name
    processing_file = PROCESSING / file_name

    print(f"\n{'=' * 55}")
    print(f"PROCESSING: {file_name} ({source_file.stat().st_size} bytes)")
    print(f"{'=' * 55}")

    # 1. Lock: move file from landing/ to processing/
    move_file(source_file, processing_file)

    try:
        # 2. Ingest
        df = load_dataframe(processing_file)
        if df is None:
            print(f"FAILED: Could not parse dataframe for {file_name}")
            move_file(processing_file, ERROR / file_name)
            return

        if len(df.columns) == 0:
            print(f"WARNING: Dataset has 0 columns: {file_name}")
            move_file(processing_file, ERROR / file_name)
            return

        # 3. Profile
        data_types, column_profile, duplicates = profile_dataframe(df, file_name)

        # 4. Prompts
        c_prompt, s_prompt = build_prompts(file_name, df, data_types, column_profile)

        # 5. Two AI passes
        print(f"Requesting AI Data Governance & Classification ({MODEL})...")
        classification_json = ask_gemini(c_prompt)

        print(f"Requesting Target Schema & Key Recommendations ({MODEL})...")
        schema_json = ask_gemini(s_prompt)

        # 6. Save outputs
        stem = Path(file_name).stem
        classification_file = OUTPUT / f"{stem}_classification.json"
        schema_file = OUTPUT / f"{stem}_schema.json"
        processed_file = OUTPUT / f"{stem}_processed.csv"

        classification_file.write_text(classification_json, encoding="utf-8")
        schema_file.write_text(schema_json, encoding="utf-8")
        df.to_csv(processed_file, index=False)

        # 7. Archive the source file
        move_file(processing_file, ARCHIVE / file_name)

    except Exception as e:
        print(f"ERROR processing {file_name}: {e}")
        move_file(processing_file, ERROR / file_name)
        return

    print(f"\nSUCCESS: {file_name}")
    print(f"  Profile:        {OUTPUT / f'{stem}_profile.csv'}")
    print(f"  Classification: {classification_file}")
    print(f"  Schema:         {schema_file}")
    print(f"  Processed CSV:  {processed_file}")
    print(f"  Archived to:    {ARCHIVE / file_name}")


def main():
    """
    Batch scanner:
    - scans landing/ for .csv, .json, .xlsx, .txt
    - quarantines unsupported formats into error/
    - passes each valid file to process_file()
    """
    supported_extensions = [".csv", ".json", ".xlsx", ".txt"]
    files = []

    for f in LANDING.iterdir():
        if f.is_file():
            if f.suffix.lower() in supported_extensions:
                files.append(f)
            else:
                print(f"Unsupported file quarantined: {f.name}")
                move_file(f, ERROR / f.name)

    files.sort()
    print("\n--- Batch Ingestion Scan ---")
    print(f"Found {len(files)} supported file(s) in {LANDING}:")
    for f in files:
        print(f"  -> {f.name}")

    if not files:
        print("No files available in landing/ to process.")
        return

    for source_file in files:
        process_file(source_file)

    print(f"\nAll files processed! Check {OUTPUT} and {ARCHIVE}.")

## 7. Run the batch

main()

## 8. Check the results
print("OUTPUT:")
display(dbutils.fs.ls(str(OUTPUT)))

print("ARCHIVE:")
display(dbutils.fs.ls(str(ARCHIVE)))

print("ERROR:")
display(dbutils.fs.ls(str(ERROR)))